# submission


In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project


Mounted at /content/drive
/content/drive/MyDrive/ml_final_project


In [2]:
from colab_setup import setup_project

drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")

import feature_pipeline
import importlib
importlib.reload(feature_pipeline)
from feature_pipeline import load_raw_data, run_feature_pipeline, split_and_save

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212

In [4]:
import mlflow
import dagshub

dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d471d7fb-73d9-420f-8ce4-0ff6921ddd60&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=d8327f8634ab6adb1cc3b9f0b4f9de1a2366504a3d13f44583b8051507583dd4




Accessing as aochi23

Initialized MLflow to track repo "aochi23/ml_final_project"

Repository aochi23/ml_final_project initialized!

In [23]:
train, test, features, stores = load_raw_data(path=f'{drive_repo}/data/')

raw_test = test[['Store', 'Dept', 'Date', 'IsHoliday']].copy()

full_df = run_feature_pipeline(train, test, features, stores)
_, processed_test = split_and_save(full_df)

processed_test['Store'] = processed_test['Store'].astype('category')
processed_test['Dept'] = processed_test['Dept'].astype('category')

print(f"raw_test: {raw_test.shape}")
print(f"processed_test: {processed_test.shape}")

raw_test: (115064, 4)
processed_test: (115064, 38)


In [8]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

all_experiments = client.search_experiments()
print(f"Found {len(all_experiments)} experiments:")
for e in all_experiments:
    print(f"  {e.name} (id={e.experiment_id})")

Found 10 experiments:
  TFT_Training (id=9)
  PatchTST_Training (id=8)
  NBEATS_Training (id=7)
  TimesFM_Training (id=6)
  DLinear_Training (id=5)
  LightGBM_Training (id=4)
  ARIMA_Training (id=3)
  XGBoost_Training (id=2)
  Prophet_Training (id=1)
  SARIMA_Training (id=0)


In [9]:
def get_model_id_from_run(run_id):
    run = mlflow.get_run(run_id)
    try:
        model_outputs = run.outputs.model_outputs
        if model_outputs:
            return model_outputs[0].model_id
    except AttributeError:
        pass

    try:
        logged_models = client.search_logged_models(
            experiment_ids=[run.info.experiment_id],
            filter_string=f"source_run_id = '{run_id}'",
        )
        if logged_models:
            return logged_models[0].model_id
    except Exception:
        pass

    return None

In [16]:
def find_best_model_uri(experiment_id, experiment_name, validate=True):
    runs = mlflow.search_runs(
        experiment_ids=[experiment_id],
        order_by=["metrics.val_wmae ASC"],
        max_results=50,
    )
    if runs.empty:
        return None, None, "no runs found"

    for _, run_row in runs.iterrows():
        run_id = run_row['run_id']
        model_id = get_model_id_from_run(run_id)
        if not model_id:
            continue
        uri = f"models:/{model_id}"
        if validate:
            try:
                mlflow.pyfunc.load_model(uri)
            except Exception as e:
                print(f"    skipping broken model {uri}: {type(e).__name__}")
                continue
        wmae = run_row.get('metrics.val_wmae', None)
        run_name = run_row.get('tags.mlflow.runName', run_id)
        return uri, wmae, f"run={run_name}"

    return None, None, "no valid logged model found"


discovered_uris = {}
discovered_wmae = {}

for exp in all_experiments:
    if exp.name == "Default":
        continue
    model_name = exp.name.replace("_Training", "")
    uri, wmae, info = find_best_model_uri(exp.experiment_id, exp.name)
    if uri:
        discovered_uris[model_name] = uri
        discovered_wmae[model_name] = wmae
        print(f"{model_name}: {uri}  (val_wmae={wmae})  [{info}]")
    else:
        print(f"{model_name}: NOT FOUND — {info}")

2026/07/12 11:00:30 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - pytorch-forecasting (current: uninstalled, required: pytorch-forecasting==1.8.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


    skipping broken model models:/m-d396503f809247029878592e1c40c5ef: ModuleNotFoundError


2026/07/12 11:00:31 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at https://dagshub.com/aochi23/ml_final_project.mlflow/api/2.0/mlflow-artifacts/artifacts/3f24e094d54c468cbb5545522a77e88a/models/m-7c0234f5e10244c2bffaf40abd6b8a3c/artifacts. Returning destination path.


    skipping broken model models:/m-7c0234f5e10244c2bffaf40abd6b8a3c: MlflowException


2026/07/12 11:00:31 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at https://dagshub.com/aochi23/ml_final_project.mlflow/api/2.0/mlflow-artifacts/artifacts/3f24e094d54c468cbb5545522a77e88a/models/m-0688fc94f13a4157a74721141f3d09ae/artifacts. Returning destination path.


...

    skipping broken model models:/m-8076ff0b8dda4498bae00ad66bf0a419: MlflowException


2026/07/12 11:01:54 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at https://dagshub.com/aochi23/ml_final_project.mlflow/api/2.0/mlflow-artifacts/artifacts/b5a37f381b794c87a5f0b7d8d6ea4253/models/m-03a8699949144b08b43bc81529df35bf/artifacts. Returning destination path.


    skipping broken model models:/m-03a8699949144b08b43bc81529df35bf: MlflowException


2026/07/12 11:01:55 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at https://dagshub.com/aochi23/ml_final_project.mlflow/api/2.0/mlflow-artifacts/artifacts/b5a37f381b794c87a5f0b7d8d6ea4253/models/m-9dba2daeac9944939e0e015e68183fe8/artifacts. Returning destination path.


    skipping broken model models:/m-9dba2daeac9944939e0e015e68183fe8: MlflowException
SARIMA: NOT FOUND — no valid logged model found


In [11]:
print("Tracking URI:", mlflow.get_tracking_uri())
print("discovered_uris exists and has:", len(discovered_uris) if 'discovered_uris' in dir() else "NOT DEFINED")
print("all_experiments exists and has:", len(all_experiments) if 'all_experiments' in dir() else "NOT DEFINED")

Tracking URI: https://dagshub.com/aochi23/ml_final_project.mlflow
discovered_uris exists and has: 6
all_experiments exists and has: 10


In [12]:
import torch

_original_torch_load = torch.load
def _cpu_safe_load(*args, **kwargs):
    if not torch.cuda.is_available():
        kwargs.setdefault('map_location', torch.device('cpu'))
    return _original_torch_load(*args, **kwargs)
torch.load = _cpu_safe_load

In [15]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class XGBFeaturePrep(BaseEstimator, TransformerMixin):
    def __init__(self, feature_cols):
        self.feature_cols = feature_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['yoy_growth_v2'] = (
            (X['lag_52'] - X['lag_53']) / X['lag_53'].replace(0, np.nan)
        )
        X['Store'] = X['Store'].astype('category')
        X['Dept'] = X['Dept'].astype('category')
        return X[self.feature_cols]

In [17]:
print("Successfully loaded models:")
for name, uri in discovered_uris.items():
    print(f"{name}: {uri}")

Successfully loaded models:
PatchTST: models:/m-6e594f0c6a0b47fab1d5c7299876d404
NBEATS: models:/m-3f676ed18ea544529fb13a0b647bd61d
DLinear: models:/m-240bbfbb71224e21b15399c328e109f4
LightGBM: models:/m-b47ac5ae45244bc9ba7d15b5f62e9f48
ARIMA: models:/m-32119e73e73c466dbc57a5b3e40ab874
XGBoost: models:/m-55ebed8898d94128829ae8f4e111d739
Prophet: models:/m-b728d29c334b421bbbf9a90a72c752d4


In [18]:
loaded_models = {}
load_failures = {}

for name, uri in discovered_uris.items():
    if 'TFT' in name:
        print(f"Skipping {name} (handling separately)")
        continue
    try:
        model = mlflow.pyfunc.load_model(uri)
        loaded_models[name] = model
        print(f"Loaded: {name}")
    except Exception as e:
        load_failures[name] = str(e)
        print(f"FAILED to load {name}: {type(e).__name__}: {e}")

print(f"\nTotal loaded: {len(loaded_models)} / {len(discovered_uris) - 1} (excluding TFT)")
if load_failures:
    print("\nLoad failures:")
    for k, v in load_failures.items():
        print(f"  {k}: {v}")

Loaded: PatchTST


Loaded: NBEATS


Loaded: DLinear


2026/07/12 11:02:54 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - dask (current: 2026.1.1, required: dask==2026.6.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


Loaded: LightGBM


Loaded: ARIMA


Loaded: XGBoost


Loaded: Prophet

Total loaded: 7 / 6 (excluding TFT)


In [24]:
raw_test = test[['Store', 'Dept', 'Date', 'IsHoliday']].copy()

processed_test_xgb = processed_test.copy()
processed_test_xgb['Store'] = processed_test_xgb['Store'].astype('category')
processed_test_xgb['Dept']  = processed_test_xgb['Dept'].astype('category')


raw_input_models  = {'PatchTST', 'NBEATS', 'DLinear', 'LightGBM'}
proc_input_models = {'XGBoost', 'Prophet', 'ARIMA'}

results = {}

for name, model in loaded_models.items():
    print(f"\n--- {name} ---")
    try:
        if name in raw_input_models:
            input_data = raw_test
        elif name == 'XGBoost':
            input_data = processed_test_xgb
        else:
            input_data = processed_test

        preds = model.predict(input_data)
        preds = np.asarray(preds).flatten()
        preds = np.clip(preds, a_min=0, a_max=None)

        results[name] = preds
        print(f"  OK — {len(preds)} predictions")
        print(f"  mean={preds.mean():.2f}, min={preds.min():.2f}, max={preds.max():.2f}")

    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")

print(f"\nSuccessful: {list(results.keys())}")


--- PatchTST ---
  OK — 115064 predictions
  mean=15311.23, min=0.00, max=223235.55

--- NBEATS ---
  OK — 115064 predictions
  mean=14268.98, min=0.00, max=189744.62

--- DLinear ---
  OK — 115064 predictions
  mean=14156.69, min=0.00, max=181877.06

--- LightGBM ---
  OK — 115064 predictions
  mean=16715.03, min=0.00, max=236086.57

--- ARIMA ---


/tmp/ipykernel_2607/2899810015.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


  OK — 115064 predictions
  mean=15982.84, min=22.39, max=53399.05

--- XGBoost ---
  OK — 115064 predictions
  mean=16753.82, min=0.00, max=527706.62

--- Prophet ---


/tmp/ipykernel_1429/2370345757.py:48: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


  OK — 115064 predictions
  mean=16868.95, min=0.00, max=649267.58

Successful: ['PatchTST', 'NBEATS', 'DLinear', 'LightGBM', 'ARIMA', 'XGBoost', 'Prophet']


In [ ]:
import os
import pandas as pd

val_scores = {
    'NBEATS':    1263.22,
    'LightGBM':  1268.73,
    'PatchTST':  1305.83,
    'Prophet':   1411.22,
    'XGBoost':   1442.88,
    'DLinear':   1515.11,
    'ARIMA':     2187.49,
}

submission_paths = {}

for name, preds in results.items():
    sub = test[['Store', 'Dept', 'Date']].copy()
    sub['Date'] = pd.to_datetime(sub['Date'])
    sub['Id'] = (
        sub['Store'].astype(str) + '_' +
        sub['Dept'].astype(str) + '_' +
        sub['Date'].dt.strftime('%Y-%m-%d')
    )
    sub['Weekly_Sales'] = np.clip(preds, 0, None)
    sub = sub[['Id', 'Weekly_Sales']]

    path = f"{drive_repo}/submission_{name}.csv"
    sub.to_csv(path, index=False)
    submission_paths[name] = path
    print(f"Saved {name}: {len(sub)} rows, val WMAE={val_scores.get(name, 'N/A')}")

print(f"\nAll submissions saved. Recommended Kaggle submission order:")
for name in sorted(submission_paths.keys(), key=lambda x: val_scores.get(x, 9999)):
    print(f"  {name}: val WMAE={val_scores.get(name, 'N/A')}")